# 04 — End-to-End Validation: Food Scan Analyzer

**Simulasi full pipeline:** nama makanan dari Gemini Vision → match → nutrisi → health score

**Test scenarios:**
1. Sarapan sehat (WEIGHT_LOSS)
2. Makan siang binaragawan (MUSCLE_GAIN)
3. Snack tinggi kalori (WEIGHT_LOSS — harapan: POOR)
4. Makan penderita diabetes (kondisi khusus)
5. Makanan asing OOD: sushi/ramen/pizza

**Output:** `output/validation/validation_report.json`

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import os
import unicodedata
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

os.makedirs('output/validation', exist_ok=True)

# Load semua artifacts
vectorizer  = joblib.load('output/models/food_tfidf_vectorizer.pkl')
food_matrix = np.load('output/models/food_name_matrix.npy')
scorer      = joblib.load('output/models/nutrition_scorer.pkl')
df_food     = pd.read_parquet('output/preprocessed/food_processed.parquet')

with open('output/preprocessed/alias_map.json', encoding='utf-8') as f:
    alias_map = json.load(f)
with open('output/models/scanner_config.json') as f:
    config = json.load(f)

print(f'Artifacts loaded:')
print(f'  Food database: {len(df_food)} items')
print(f'  Food matrix:   {food_matrix.shape}')
print(f'  Alias map:     {len(alias_map)} entries')
print(f'  Scorer F1:     {config["f1_macro"]}')

Artifacts loaded:
  Food database: 1346 items
  Food matrix:   (1346, 5141)
  Alias map:     97 entries
  Scorer F1:     0.9567


In [2]:
# ── Helper Functions ──
def normalize_food_name(text):
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

LABEL_MAP = config['label_map']  # {'0': 'POOR', '1': 'MODERATE', '2': 'GOOD'}
GOAL_MAP  = {v: int(k) for k, v in config['goal_map'].items()}
COND_MAP  = config['condition_map']

def analyze_food_scan(identified_foods, user_goal='WEIGHT_LOSS',
                       user_condition='None', portions=None):
    """
    Full pipeline: food names (dari Gemini Vision) -> match -> nutrition -> health score.
    """
    portions = portions or [1.0] * len(identified_foods)
    matches  = []
    total_cal, total_protein, total_fat, total_carbs = 0.0, 0.0, 0.0, 0.0

    for food_name, portion in zip(identified_foods, portions):
        norm = normalize_food_name(food_name)
        # Apply alias mapping
        if norm in alias_map:
            norm = alias_map[norm]

        # TF-IDF cosine similarity
        q_vec = vectorizer.transform([norm])
        sims  = cosine_similarity(q_vec, food_matrix).flatten()
        best_idx   = int(np.argmax(sims))
        confidence = float(sims[best_idx])

        if confidence >= 0.20:
            row = df_food.iloc[best_idx]
            cal = float(row['calories_per_portion']) * portion
            matches.append({
                'query':      food_name,
                'matched':    row['name'],
                'confidence': round(confidence, 4),
                'calories':   round(cal, 1),
                'protein_g':  round(float(row['protein_g']) * portion, 1),
                'fat_g':      round(float(row['fat_g']) * portion, 1),
                'carbs_g':    round(float(row['carbs_g']) * portion, 1),
                'category':   row['category'],
                'is_halal':   bool(row['is_halal']),
            })
            total_cal     += cal
            total_protein += float(row['protein_g']) * portion
            total_fat     += float(row['fat_g'])     * portion
            total_carbs   += float(row['carbs_g'])   * portion
        else:
            matches.append({
                'query':      food_name,
                'matched':    None,
                'confidence': round(confidence, 4),
                'note':       'Tidak dikenali — coba ketik manual',
            })

    # Health scoring
    goal_enc = GOAL_MAP.get(user_goal, 0)
    cond_enc = COND_MAP.get(user_condition, 0)
    X_input = np.array([[total_cal, total_protein, total_fat, total_carbs,
                         len(identified_foods), goal_enc, cond_enc]])
    pred_label = int(scorer.predict(X_input)[0])
    pred_proba = scorer.predict_proba(X_input)[0]
    assessment = LABEL_MAP[str(pred_label)]

    return {
        'matches':         matches,
        'nutrition_total': {
            'calories':  round(total_cal, 1),
            'protein_g': round(total_protein, 1),
            'fat_g':     round(total_fat, 1),
            'carbs_g':   round(total_carbs, 1),
        },
        'health_score':    round(float(max(pred_proba)), 4),
        'assessment':      assessment,
        'user_goal':       user_goal,
        'user_condition':  user_condition,
    }

print('Pipeline function siap.')

Pipeline function siap.


In [3]:
# ── End-to-End Test Scenarios ──
SCENARIOS = [
    {
        'name': '1. Sarapan sehat (Weight Loss)',
        'foods': ['nasi putih', 'telur rebus', 'bayam rebus'],
        'goal': 'WEIGHT_LOSS', 'condition': 'None',
        'expected_assessment': ['GOOD', 'MODERATE'],
    },
    {
        'name': '2. Makan siang binaragawan (Muscle Gain)',
        'foods': ['nasi merah', 'ayam goreng', 'brokoli'],
        'goal': 'MUSCLE_GAIN', 'condition': 'None',
        'expected_assessment': ['GOOD', 'MODERATE'],
    },
    {
        'name': '3. Snack tinggi kalori (Weight Loss)',
        'foods': ['kerupuk kulit kerbau', 'es teh'],
        'goal': 'WEIGHT_LOSS', 'condition': 'None',
        'expected_assessment': ['POOR', 'MODERATE'],
    },
    {
        'name': '4. Makan penderita diabetes',
        'foods': ['nasi goreng', 'tahu goreng'],
        'goal': 'MAINTENANCE', 'condition': 'Diabetes',
        'expected_assessment': ['POOR', 'MODERATE'],
    },
    {
        'name': '5. OOD: makanan asing',
        'foods': ['sushi', 'ramen', 'pizza'],
        'goal': 'MAINTENANCE', 'condition': 'None',
        'expected_no_match': True,
    },
]

print('=' * 60)
print('END-TO-END VALIDATION — Food Scan Analyzer')
print('=' * 60)

all_results = []
for scenario in SCENARIOS:
    result = analyze_food_scan(
        scenario['foods'], scenario['goal'], scenario.get('condition', 'None')
    )

    # Display
    print(f"\n{scenario['name']}")
    print(f"  Input:      {scenario['foods']}")
    for m in result['matches']:
        if m.get('matched'):
            print(f"  Match:      '{m['query']}' -> '{m['matched']}' (conf: {m['confidence']:.3f})")
        else:
            print(f"  No match:   '{m['query']}' (conf: {m['confidence']:.3f}) [OOD]")
    print(f"  Nutrition:  cal={result['nutrition_total']['calories']} kcal, "
          f"protein={result['nutrition_total']['protein_g']}g, "
          f"fat={result['nutrition_total']['fat_g']}g")
    print(f"  Assessment: {result['assessment']} (score: {result['health_score']:.3f})")

    all_results.append({'scenario': scenario['name'], **result})

END-TO-END VALIDATION — Food Scan Analyzer

1. Sarapan sehat (Weight Loss)
  Input:      ['nasi putih', 'telur rebus', 'bayam rebus']
  Match:      'nasi putih' -> 'Nasi' (conf: 0.634)
  Match:      'telur rebus' -> 'Kacang Telur ' (conf: 0.741)
  Match:      'bayam rebus' -> 'Bayam segar' (conf: 0.810)
  Nutrition:  cal=716.0 kcal, protein=20.4g, fat=14.7g
  Assessment: POOR (score: 0.999)

2. Makan siang binaragawan (Muscle Gain)
  Input:      ['nasi merah', 'ayam goreng', 'brokoli']
  Match:      'nasi merah' -> 'Nasi beras merah' (conf: 0.827)
  Match:      'ayam goreng' -> 'Ayam usus goreng' (conf: 0.726)
  Match:      'brokoli' -> 'Brondong ' (conf: 0.348)
  Nutrition:  cal=1022.0 kcal, protein=52.0g, fat=27.4g
  Assessment: GOOD (score: 0.998)

3. Snack tinggi kalori (Weight Loss)
  Input:      ['kerupuk kulit kerbau', 'es teh']
  Match:      'kerupuk kulit kerbau' -> 'Kerupuk Kulit kerbau' (conf: 0.945)
  Match:      'es teh' -> 'Teh' (conf: 0.609)
  Nutrition:  cal=532.0 kcal,

In [4]:
# ── Save Validation Report ──
match_rates = []
for r, s in zip(all_results, SCENARIOS):
    if s.get('expected_no_match'):
        # OOD: count unmatched as success
        unmatched = sum(1 for m in r['matches'] if not m.get('matched'))
        match_rates.append(unmatched / len(r['matches']))
    else:
        matched = sum(1 for m in r['matches'] if m.get('matched'))
        match_rates.append(matched / len(r['matches']))

report = {
    'version':             '1.0',
    'n_scenarios':         len(all_results),
    'avg_match_rate':      round(float(np.mean(match_rates)), 4),
    'target_match_rate':   0.80,
    'pass':                float(np.mean(match_rates)) >= 0.80,
    'per_scenario': [
        {
            'name':             r['scenario'],
            'match_rate':       round(match_rates[i], 4),
            'assessment':       r['assessment'],
            'total_calories':   r['nutrition_total']['calories'],
            'n_foods_input':    len(r['matches']),
            'n_foods_matched':  sum(1 for m in r['matches'] if m.get('matched')),
        }
        for i, r in enumerate(all_results)
    ],
    'artifacts': {
        'vectorizer':     'output/models/food_tfidf_vectorizer.pkl',
        'food_matrix':    'output/models/food_name_matrix.npy',
        'scorer':         'output/models/nutrition_scorer.pkl',
        'scanner_config': 'output/models/scanner_config.json',
        'alias_map':      'output/preprocessed/alias_map.json',
        'food_db':        'output/preprocessed/food_processed.parquet',
    }
}
with open('output/validation/validation_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(f'\n=== SUMMARY ===')
print(f'Avg match rate: {report["avg_match_rate"]*100:.1f}% (target >= 80%)')
print(f'Status: {"PASS" if report["pass"] else "BELOW TARGET"}')
print('\nValidation report saved -> output/validation/validation_report.json')

# FastAPI endpoint note
print("""
=== FastAPI Endpoint Reference ===
POST /predict/food-scan
Body: {
  "identified_foods": ["nasi goreng", "telur"],   # dari Gemini Vision
  "user_goal": "WEIGHT_LOSS",
  "user_condition": "None",
  "portions": [1, 2]
}
Response: {
  "matches": [...],
  "nutrition_total": {...},
  "health_score": 0.82,
  "assessment": "GOOD"
}
""")


=== SUMMARY ===
Avg match rate: 86.7% (target >= 80%)
Status: PASS

Validation report saved -> output/validation/validation_report.json

=== FastAPI Endpoint Reference ===
POST /predict/food-scan
Body: {
  "identified_foods": ["nasi goreng", "telur"],   # dari Gemini Vision
  "user_goal": "WEIGHT_LOSS",
  "user_condition": "None",
  "portions": [1, 2]
}
Response: {
  "matches": [...],
  "nutrition_total": {...},
  "health_score": 0.82,
  "assessment": "GOOD"
}

